<a href="https://colab.research.google.com/github/Noadori34/Web-analytics-project/blob/main/Project_Crawling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import time
import requests
import pandas as pd
from google.colab import userdata

# 1. Technical Skill Taxonomy
SKILL_TAXONOMY = {
    "Languages": ["python", "r", "sql", "java", "scala", "c++", "typescript", "go", "rust"],
    "Cloud & DevOps": ["aws", "azure", "gcp", "docker", "kubernetes", "terraform", "ci/cd"],
    "Data & AI": ["spark", "hadoop", "airflow", "snowflake", "dbt", "pytorch", "tensorflow", "scikit-learn", "tableau", "power bi"],
    "Web/Frameworks": ["react", "node.js", "django", "fastapi", "flask"]
}

def clean_html(raw_text: str) -> str:
    """Removes raw HTML formatting tags and collapses extra whitespace."""
    if not raw_text:
        return ""
    clean_text = re.sub(r'<[^>]+>', ' ', raw_text)
    return " ".join(clean_text.split())

def extract_work_model_us(text: str) -> str:
    """Classifies US job listings into Remote, Hybrid, or On-site."""
    text_lower = text.lower()
    if "hybrid" in text_lower:
        return "Hybrid"
    elif any(k in text_lower for k in ["remote", "work from home", "telecommute", "wfh"]):
        return "Remote"
    return "On-site"

def extract_skills(text: str) -> list:
    """Extracts technical skills using regex word boundaries."""
    text_lower = text.lower()
    detected = []
    for category, skills in SKILL_TAXONOMY.items():
        for skill in skills:
            pattern = r'\b' + skill + r'\b'
            if re.search(pattern, text_lower):
                detected.append(skill.replace('\\', '').upper())
    return list(set(detected))

def fetch_adzuna_us_jobs_robust(api_id: str, api_key: str, role: str = "data engineer", pages: int = 5) -> pd.DataFrame:
    records = []
    # Endpoint configured for US listings
    base_url = "https://api.adzuna.com/v1/api/jobs/us/search/"

    for page in range(1, pages + 1):
        params = {
            "app_id": api_id,
            "app_key": api_key,
            "results_per_page": 50,
            "what": role,
            "content-type": "application/json"
        }

        try:
            response = requests.get(f"{base_url}{page}", params=params, timeout=10)
            if response.status_code != 200:
                print(f"Skipping page {page}: API returned HTTP {response.status_code}")
                continue
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Network error on page {page}: {e}")
            continue

        for job in data.get("results", []):
            title = job.get("title") or ""
            raw_desc = job.get("description") or ""
            clean_desc = clean_html(raw_desc)
            combined_text = f"{title} {clean_desc}"

            # Null-safe dictionary handling for missing values
            company_info = job.get("company") or {}
            location_info = job.get("location") or {}

            # Average annual salary in USD
            sal_min = job.get("salary_min")
            sal_max = job.get("salary_max")
            avg_salary = (sal_min + sal_max) / 2 if (sal_min is not None and sal_max is not None) else None

            records.append({
                "job_id": job.get("id"),
                "title": clean_html(title),
                "company": company_info.get("display_name", "Unknown"),
                "city_state": location_info.get("display_name", "Unknown"),
                "area_hierarchy": location_info.get("area", []), # e.g. ['US', 'California', 'San Francisco']
                "work_model": extract_work_model_us(combined_text),
                "salary_usd": avg_salary,
                "is_predicted_salary": job.get("salary_is_predicted"),
                "created_date": job.get("created"),
                "description_snippet": clean_desc,
                "redirect_url": job.get("redirect_url"),
            })

        time.sleep(0.5) # Polite API spacing

    df = pd.DataFrame(records)
    return df, data

# Example Execution
if __name__ == "__main__":
    APP_ID = userdata.get('APP_ID')
    APP_KEY = userdata.get('APP_KEY')

    # Run fetcher for US job listings
    df_us, data = fetch_adzuna_us_jobs_robust(APP_ID, APP_KEY, role="data science", pages=45)

    print(f"Successfully fetched {len(df_us)} US job listings.")
    if not df_us.empty:
        print(df_us[["title", "city_state", "work_model", "salary_usd"]].head())

Successfully fetched 2250 US job listings.


KeyError: "['extracted_skills'] not in index"

In [20]:
'https://www.adzuna.com/details/' + df_us.iloc[1]['redirect_url'].split('/')[-1]

'https://www.adzuna.com/details/5929942144?se=pBAjbeTd8RGc2vUBhOaUNg&utm_medium=api&utm_source=efdde0f2&v=880A0CE6833DDC0D4C26146F578BA68B057999E3'

In [12]:
data_frame = pd.DataFrame(data['results'])

In [5]:
df_us.to_csv('adzuna_data_more.csv', index=False, encoding='utf-8-sig')
data.to_json("adzuna_data_full.json", index=False, encoding='utf-8-sig')

In [6]:
import json

with open("adzuna_data_full.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

In [21]:
def url_reconstruction(url: str) -> float:
    id = url.split('/')[-1]
    new_url = 'https://www.adzuna.com/details/' + id
    return new_url

In [44]:
df_us.head()

,job_id,title,company,city_state,area_hierarchy,work_model,salary_usd,is_predicted_salary,created_date,description_snippet,redirect_url
0,5947302622,"Director, Data Science",CarMax,"Ridge, Henrico County","[US, Virginia, Henrico County, Ridge]",On-site,135573.00,1,2026-10-30T18:55:30Z,8901 - Corp Office West Crk - 12800 Tuckahoe C...,https://www.adzuna.com/land/ad/5947302622?se=p...
1,5929942144,Academic Surgeon - Surgical Data Science (Assi...,Medical College of Wisconsin,"Elm Grove, Waukesha County","[US, Wisconsin, Waukesha County, Elm Grove]",On-site,133762.94,1,2026-10-15T23:36:47Z,Position Summary: The Medical College of Wisco...,https://www.adzuna.com/land/ad/5929942144?se=p...
2,5952032377,Data Science,"Maxima Technologies, LLC","Lincolnshire, Lake County","[US, Illinois, Lake County, Lincolnshire]",On-site,207987.16,1,2026-11-03T13:34:12Z,Minimum Work Experience (years): 10 years of e...,https://www.adzuna.com/land/ad/5952032377?se=p...
3,5948635953,"Director, Data Science",Discover Financial Services,"Grand Central, Manhattan","[US, New York, New York City, Manhattan, Grand...",On-site,248871.06,1,2026-11-01T00:40:30Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948635953?se=p...
4,5948637300,"Director, Data Science",Discover Financial Services,"Hayes Valley, San Francisco","[US, California, San Francisco, Hayes Valley]",On-site,247230.06,1,2026-11-01T00:41:54Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948637300?se=p...


In [46]:
df_us['salary_usd'].isna().value_counts()

,count
salary_usd,
False,2247
True,3


In [25]:
df_10 = df_us.copy()
df_10['new_url'] = df_10['redirect_url'].apply(url_reconstruction)

In [29]:
df_10

,job_id,title,company,city_state,area_hierarchy,work_model,salary_usd,is_predicted_salary,created_date,description_snippet,redirect_url,new_url
0,5947302622,"Director, Data Science",CarMax,"Ridge, Henrico County","[US, Virginia, Henrico County, Ridge]",On-site,135573.00,1,2026-10-30T18:55:30Z,8901 - Corp Office West Crk - 12800 Tuckahoe C...,https://www.adzuna.com/land/ad/5947302622?se=p...,https://www.adzuna.com/details/5947302622?se=p...
1,5929942144,Academic Surgeon - Surgical Data Science (Assi...,Medical College of Wisconsin,"Elm Grove, Waukesha County","[US, Wisconsin, Waukesha County, Elm Grove]",On-site,133762.94,1,2026-10-15T23:36:47Z,Position Summary: The Medical College of Wisco...,https://www.adzuna.com/land/ad/5929942144?se=p...,https://www.adzuna.com/details/5929942144?se=p...
2,5952032377,Data Science,"Maxima Technologies, LLC","Lincolnshire, Lake County","[US, Illinois, Lake County, Lincolnshire]",On-site,207987.16,1,2026-11-03T13:34:12Z,Minimum Work Experience (years): 10 years of e...,https://www.adzuna.com/land/ad/5952032377?se=p...,https://www.adzuna.com/details/5952032377?se=p...
3,5948635953,"Director, Data Science",Discover Financial Services,"Grand Central, Manhattan","[US, New York, New York City, Manhattan, Grand...",On-site,248871.06,1,2026-11-01T00:40:30Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948635953?se=p...,https://www.adzuna.com/details/5948635953?se=p...
4,5948637300,"Director, Data Science",Discover Financial Services,"Hayes Valley, San Francisco","[US, California, San Francisco, Hayes Valley]",On-site,247230.06,1,2026-11-01T00:41:54Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948637300?se=p...,https://www.adzuna.com/details/5948637300?se=p...
...,...,...,...,...,...,...,...,...,...,...,...,...
2245,5762278101,Applied Data Science & Insights Leader - GTM I...,OpenAI,"San Francisco, California","[US, California, San Francisco]",On-site,478000.00,0,2026-06-13T03:42:29Z,About the Team The GTM Data Science team partn...,https://www.adzuna.com/details/5762278101?utm_...,https://www.adzuna.com/details/5762278101?utm_...
2246,5360100470,ChemBE Systems/Data Science/Artificial Intelli...,Johns Hopkins University,"Druid, Baltimore","[US, Maryland, Baltimore County, Baltimore, Dr...",On-site,242171.60,1,2025-08-20T08:22:00Z,Faculty position in Chemical and Biomolecular ...,https://www.adzuna.com/land/ad/5360100470?se=A...,https://www.adzuna.com/details/5360100470?se=A...
2247,5955252642,"(USA) Senior Manager, Data Science - AI Strate...",Walmart,"Bentonville, Benton County","[US, Arkansas, Benton County, Bentonville]",On-site,165000.00,0,2026-11-05T23:51:52Z,Position Summary Role summary: As a Senior Man...,https://www.adzuna.com/details/5955252642?utm_...,https://www.adzuna.com/details/5955252642?utm_...
2248,5940038587,Cancer Biology Integrated Program in Cancer an...,H. Lee Moffitt Cancer Center and Research Inst...,US,[US],On-site,79984.50,0,2026-10-24T00:41:50Z,Working at Moffitt is both a career and a miss...,https://www.adzuna.com/details/5940038587?utm_...,https://www.adzuna.com/details/5940038587?utm_...


In [28]:
df_10.iloc[1]['new_url']

'https://www.adzuna.com/details/5929942144?se=pBAjbeTd8RGc2vUBhOaUNg&utm_medium=api&utm_source=efdde0f2&v=880A0CE6833DDC0D4C26146F578BA68B057999E3'

In [37]:
df_15 = df_10.head(15).copy()
df_15

,job_id,title,company,city_state,area_hierarchy,work_model,salary_usd,is_predicted_salary,created_date,description_snippet,redirect_url,new_url
0,5947302622,"Director, Data Science",CarMax,"Ridge, Henrico County","[US, Virginia, Henrico County, Ridge]",On-site,135573.00,1,2026-10-30T18:55:30Z,8901 - Corp Office West Crk - 12800 Tuckahoe C...,https://www.adzuna.com/land/ad/5947302622?se=p...,https://www.adzuna.com/details/5947302622?se=p...
1,5929942144,Academic Surgeon - Surgical Data Science (Assi...,Medical College of Wisconsin,"Elm Grove, Waukesha County","[US, Wisconsin, Waukesha County, Elm Grove]",On-site,133762.94,1,2026-10-15T23:36:47Z,Position Summary: The Medical College of Wisco...,https://www.adzuna.com/land/ad/5929942144?se=p...,https://www.adzuna.com/details/5929942144?se=p...
2,5952032377,Data Science,"Maxima Technologies, LLC","Lincolnshire, Lake County","[US, Illinois, Lake County, Lincolnshire]",On-site,207987.16,1,2026-11-03T13:34:12Z,Minimum Work Experience (years): 10 years of e...,https://www.adzuna.com/land/ad/5952032377?se=p...,https://www.adzuna.com/details/5952032377?se=p...
3,5948635953,"Director, Data Science",Discover Financial Services,"Grand Central, Manhattan","[US, New York, New York City, Manhattan, Grand...",On-site,248871.06,1,2026-11-01T00:40:30Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948635953?se=p...,https://www.adzuna.com/details/5948635953?se=p...
4,5948637300,"Director, Data Science",Discover Financial Services,"Hayes Valley, San Francisco","[US, California, San Francisco, Hayes Valley]",On-site,247230.06,1,2026-11-01T00:41:54Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948637300?se=p...,https://www.adzuna.com/details/5948637300?se=p...
5,5948635693,"Director, Data Science",Discover Financial Services,"Pimmit, Fairfax County","[US, Virginia, Fairfax County, Pimmit]",On-site,213312.70,1,2026-11-01T00:40:13Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948635693?se=p...,https://www.adzuna.com/details/5948635693?se=p...
6,5913748280,"Manager, Data Science",american express,"Grand Central, Manhattan","[US, New York, New York City, Manhattan, Grand...",On-site,141065.06,1,2026-10-06T07:15:57Z,The Model Risk Management Group (MRMG) is resp...,https://www.adzuna.com/land/ad/5913748280?se=p...,https://www.adzuna.com/details/5913748280?se=p...
7,5947729063,"Senior Manager, Logistics Data Science",CarMax,"Ridge, Henrico County","[US, Virginia, Henrico County, Ridge]",On-site,200000.00,0,2026-10-31T02:07:04Z,Lead CarMax’s Logistics Modeling team in devel...,https://www.adzuna.com/land/ad/5947729063?se=p...,https://www.adzuna.com/details/5947729063?se=p...
8,5952238053,Data Science/ML,Kforce Technology Staffing,"Minnetonka Mills, Hennepin County","[US, Minnesota, Hennepin County, Minnetonka Mi...",On-site,164753.10,1,2026-11-03T20:48:19Z,RESPONSIBILITIES: Kforce has a client in Minne...,https://www.adzuna.com/land/ad/5952238053?se=p...,https://www.adzuna.com/details/5952238053?se=p...
9,5956281454,Intern - Data Science,FedEx Office,"Collierville, Shelby County","[US, Tennessee, Shelby County, Collierville]",On-site,90035.73,1,2026-11-06T22:49:11Z,Acts independently on behalf of the department...,https://www.adzuna.com/land/ad/5956281454?se=p...,https://www.adzuna.com/details/5956281454?se=p...


In [38]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

def fetch_full_text(url: str) -> str:
    """Fetches and extracts full job description text from an Adzuna details page."""
    if not url or pd.isna(url):
        return ""

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")

            container = (soup.find("section", class_="adp-body mx-4 text-sm md:mx-0 md:text-base"))
            target_element = container if container else soup.body

            if target_element:
                paragraphs = target_element.find_all(['p', 'li', 'div'])
                text = " ".join([p.get_text(strip=True) for p in paragraphs])
                return " ".join(text.split())

        elif response.status_code in (403, 429):
            print(f"Warning: Received HTTP {response.status_code} for {url}. Request rate should be lowered.")

    except requests.RequestException as err:
        print(f"Network error fetching {url}: {err}")

    return ""

df_15['full_description'] = df_15['new_url'].apply(fetch_full_text)

In [41]:
df_15.iloc[0]['full_description']

"What?Where?SearchAdvancedDirector, Data Science❮ back to last searchCarMaxRichmond, Virginia, 23238$135,573per year - estimated?Apply for this job8901 - Corp Office West Crk - 12800 Tuckahoe Creek Parkway, Richmond, Virginia, 23238CarMax, the way your career should be!About This JobThe primary responsibility for the Pricing team is to accurately and systematically value each individual car that we buy and sell. Last year, CarMax sold over 1 million vehicles and appraised over 2 million vehicles, and Pricing directed over $20B in spend to purchase inventory. To properly value these vehicles, our team must develop and operationalize algorithms that best reflect customer preferences for each vehicle’s unique characteristics and that effectively respond to constantly changing consumer demand, wholesale auction supply, and other global/local market conditions.A Director, Core Pricing Systems, owns the pricing of vehicles in one of three key business areas:1. Retail Pricing – Initially pric

In [42]:
SKILL_PATTERNS = {
    # Programming Languages
    "Python": r"\bpython\b",
    "SQL": r"\bsql\b",
    "R": r"\b[Rr]\b",  # Matches standalone 'R' or 'r'
    "C++": r"\bc\+\+|\bcpp\b",
    "C#": r"\bc\#|\bc-sharp\b",
    "Java": r"\bjava\b",
    "JavaScript": r"\bjavascript\b|\bjs\b",
    "TypeScript": r"\btypescript\b|\bts\b",
    "Go": r"\bgolang\b|\bgo\b",
    "Rust": r"\brust\b",
    "Scala": r"\bscala\b",
    "Bash/Shell": r"\bbash\b|\bshell\b",

    # AI, ML & Data Science
    "PyTorch": r"\bpytarch\b|\bpytorch\b",
    "TensorFlow": r"\btensorflow\b|\btf\b",
    "Keras": r"\bkeras\b",
    "Scikit-Learn": r"\bscikit-learn\b|\bsklearn\b",
    "Pandas": r"\bpandas\b",
    "NumPy": r"\bnumpy\b",
    "Apache Spark": r"\bspark\b|\bpyspark\b",
    "Hadoop": r"\bhadoop\b",
    "Airflow": r"\bairflow\b",
    "Databricks": r"\bdatabricks\b",
    "LangChain": r"\blangchain\b",
    "Hugging Face": r"\bhugging\s*face\b",
    "LLM / GenAI": r"\bllm\b|\bllms\b|\bgenai\b|\bgenerative ai\b|\btransformers\b",
    "Computer Vision": r"\bcomputer vision\b|\bopencv\b",
    "NLP": r"\bnlp\b|\bnatural language processing\b",

    # Cloud & DevOps
    "AWS": r"\baws\b|\bamazon web services\b",
    "Azure": r"\bazure\b",
    "GCP": r"\bgcp\b|\bgoogle cloud\b",
    "Docker": r"\bdocker\b",
    "Kubernetes": r"\bkubernetes\b|\bk8s\b",
    "Terraform": r"\bterraform\b",
    "Linux": r"\blinux\b|\bunix\b",
    "Git": r"\bgit\b|\bgithub\b|\bgitlab\b",
    "CI/CD": r"\bci\/cd\b|\bcontinuous integration\b",

    # Databases & Storage
    "PostgreSQL": r"\bpostgres\b|\bpostgresql\b",
    "MySQL": r"\bmysql\b",
    "MongoDB": r"\bmongodb\b|\bmongo\b",
    "Redis": r"\bredis\b",
    "Snowflake": r"\bsnowflake\b",
    "BigQuery": r"\bbigquery\b",
    "Redshift": r"\bredshift\b",

    # BI & Visualization
    "Tableau": r"\btableau\b",
    "Power BI": r"\bpower bi\b|\bpowerbi\b",
    "Looker": r"\blooker\b"
}

def extract_skills(description):
    """Scans job description text and returns a list of matched skills."""
    if not isinstance(description, str) or not description:
        return []

    found_skills = []
    text_lower = description.lower()

    for skill_name, pattern in SKILL_PATTERNS.items():
        if re.search(pattern, text_lower, re.IGNORECASE):
            found_skills.append(skill_name)

    return found_skills

# 2. Extract skills as a list column
df_15['skills_list'] = df_15['full_description'].apply(extract_skills)

# 3. Create a comma-separated string column for easy display/CSV export
df_15['skills'] = df_15['skills_list'].apply(lambda x: ", ".join(x) if x else "None Specified")

# 4. View a sample of the results
df_15

,job_id,title,company,city_state,area_hierarchy,work_model,salary_usd,is_predicted_salary,created_date,description_snippet,redirect_url,new_url,full_description,skills_list,skills
0,5947302622,"Director, Data Science",CarMax,"Ridge, Henrico County","[US, Virginia, Henrico County, Ridge]",On-site,135573.00,1,2026-10-30T18:55:30Z,8901 - Corp Office West Crk - 12800 Tuckahoe C...,https://www.adzuna.com/land/ad/5947302622?se=p...,https://www.adzuna.com/details/5947302622?se=p...,"What?Where?SearchAdvancedDirector, Data Scienc...",[Python],Python
1,5929942144,Academic Surgeon - Surgical Data Science (Assi...,Medical College of Wisconsin,"Elm Grove, Waukesha County","[US, Wisconsin, Waukesha County, Elm Grove]",On-site,133762.94,1,2026-10-15T23:36:47Z,Position Summary: The Medical College of Wisco...,https://www.adzuna.com/land/ad/5929942144?se=p...,https://www.adzuna.com/details/5929942144?se=p...,What?Where?SearchAdvancedAcademic Surgeon - Su...,"[R, LLM / GenAI, Computer Vision]","R, LLM / GenAI, Computer Vision"
2,5952032377,Data Science,"Maxima Technologies, LLC","Lincolnshire, Lake County","[US, Illinois, Lake County, Lincolnshire]",On-site,207987.16,1,2026-11-03T13:34:12Z,Minimum Work Experience (years): 10 years of e...,https://www.adzuna.com/land/ad/5952032377?se=p...,https://www.adzuna.com/details/5952032377?se=p...,What?Where?SearchAdvancedData Science❮ back to...,"[Python, SQL, Pandas, NumPy, Apache Spark, Air...","Python, SQL, Pandas, NumPy, Apache Spark, Airf..."
3,5948635953,"Director, Data Science",Discover Financial Services,"Grand Central, Manhattan","[US, New York, New York City, Manhattan, Grand...",On-site,248871.06,1,2026-11-01T00:40:30Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948635953?se=p...,https://www.adzuna.com/details/5948635953?se=p...,"What?Where?SearchAdvancedDirector, Data Scienc...","[Python, R, Scala, Apache Spark, AWS]","Python, R, Scala, Apache Spark, AWS"
4,5948637300,"Director, Data Science",Discover Financial Services,"Hayes Valley, San Francisco","[US, California, San Francisco, Hayes Valley]",On-site,247230.06,1,2026-11-01T00:41:54Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948637300?se=p...,https://www.adzuna.com/details/5948637300?se=p...,"What?Where?SearchAdvancedDirector, Data Scienc...","[Python, R, Scala, Apache Spark, AWS]","Python, R, Scala, Apache Spark, AWS"
5,5948635693,"Director, Data Science",Discover Financial Services,"Pimmit, Fairfax County","[US, Virginia, Fairfax County, Pimmit]",On-site,213312.70,1,2026-11-01T00:40:13Z,Data is at the center of everything we do. As ...,https://www.adzuna.com/land/ad/5948635693?se=p...,https://www.adzuna.com/details/5948635693?se=p...,"What?Where?SearchAdvancedDirector, Data Scienc...","[Python, R, Scala, Apache Spark, AWS]","Python, R, Scala, Apache Spark, AWS"
6,5913748280,"Manager, Data Science",american express,"Grand Central, Manhattan","[US, New York, New York City, Manhattan, Grand...",On-site,141065.06,1,2026-10-06T07:15:57Z,The Model Risk Management Group (MRMG) is resp...,https://www.adzuna.com/land/ad/5913748280?se=p...,https://www.adzuna.com/details/5913748280?se=p...,"What?Where?SearchAdvancedManager, Data Science...","[Python, R, C++, Java, Scala, Apache Spark, Ha...","Python, R, C++, Java, Scala, Apache Spark, Had..."
7,5947729063,"Senior Manager, Logistics Data Science",CarMax,"Ridge, Henrico County","[US, Virginia, Henrico County, Ridge]",On-site,200000.00,0,2026-10-31T02:07:04Z,Lead CarMax’s Logistics Modeling team in devel...,https://www.adzuna.com/land/ad/5947729063?se=p...,https://www.adzuna.com/details/5947729063?se=p...,"What?Where?SearchAdvancedSenior Manager, Logis...",[],None Specified
8,5952238053,Data Science/ML,Kforce Technology Staffing,"Minnetonka Mills, Hennepin County","[US, Minnesota, Hennepin County, Minnetonka Mi...",On-site,164753.10,1,2026-11-03T20:48:19Z,RESPONSIBILITIES: Kforce has a client in Minne...,https://www.adzuna.